# Week 12 Lab

## Maximum Likelihood Estimation of a Normal Regression Model

## Learning Objectives

By the end of this lab, you will be able to:
1. Extend a data generating process (DGP) to include additional regressors
2. Write and implement the log-likelihood function for a normal regression model
3. Use closures and numerical optimization to find the MLE
4. Compute standard errors analytically and compare with numerical Hessian approximations

## The Normal Regression Model

In assignment 11 you have studied the **normal regression model**

\begin{align*}
    Y_i &= X_i'\beta + e_i, \qquad \text{where }
    e_i | X_i \sim \mathcal{N}(0, \sigma_e^2).
\end{align*}

As you can see, the errors here are assumed to have an **exact** normal distribution. This enables us to use maximum likelihood estimation, because we know the distribution of $Y_i$ **conditional on $X_i$**.

What are we estimating? The unknown parameters are $\beta \in \mathbb{R}^K$ and $\sigma_e^2$.

That conditional density is
\begin{align*}
    f_{Y|X}(y | x, \beta, \sigma_e^2) = \frac{1}
        {\sqrt{ 2 \pi \sigma_e^2}} \exp \left( - \frac{1}{2 \sigma_e^2} (y - x'\beta)^2
        \right).
\end{align*}

## Loading packages for this notebook

We will be needing the following packages:

In [ ]:
using LinearAlgebra, Distributions, Random, Optim

## Exercise 1

Here is an extension of the DGP from weeks 8 and 9, where we simulated Card's schooling model. The new addition is work experience $W$ with coefficient $\beta_4$:

$$
\begin{align*}
    u & \sim \mathcal{N}(0, \sigma_u^2) \\
    A & \sim \mathcal{N}(0, \sigma_A^2) && \text{(ability)}\\
    S &= \pi + A && \text{(schooling)}\\
    W & \sim \text{Discrete uniform on }\{0,1,\ldots,20\}  && \text{(work experience)} \\
    Y &= \exp (\beta_1 + \beta_2 S + \beta_3 A + \beta_4 W + u) && \text{(earnings)}
\end{align*}
$$

### What's different from weeks 8 and 9?

* Work experience $W$ is added as a new explanatory variable, with coefficient $\beta_4$ (the return to work experience).

* We set $\beta_3 = 0$, which removes the effect of ability on earnings and eliminates the endogeneity problem from week 9.

This model is a normal regression model when you consider the logarithm of $Y$:
$$\log Y = \beta_1 + \beta_2 S + \beta_3 A + \beta_4 W + u$$

With $\beta_3 = 0$, the error $u$ is independent of the regressors $(S, W)$, and we have a clean normal regression model in $\log Y$.

### Your task

Below is the `schooling_sample` function from week 9. Modify it to include:
1. A new positional argument `β4` (the return to experience)
2. Drawing experience $W$ from a discrete uniform distribution on $\{0, 1, \ldots, 20\}$
3. Adding $\beta_4 W$ to the log-earnings equation
4. Returning `S, W, Y` (three values instead of two)

Then create a random sample of size 5,000 using these calibrated values:

|                           | Calibrated values  |
|---------------------------|--------------------|
| $\beta_1$                 | 4.70               |
| $\beta_2$                 | 0.07               |
| $\beta_3$                 | 0                  |
| $\beta_4$                 | 0.12               |
| $\pi$                     | 13.2               |
| $\sigma_u^2$              | 0.175              |
| $\sigma_A^2$              | 7.20               |

### The `schooling_sample` function from week 9

Here is the week 9 version for reference. It takes `β2`, `β3`, and `N` as positional arguments and returns schooling `S` and log earnings `Y`.

```julia
function schooling_sample(β2::Real, β3::Real, N::Integer;
                          π::Real=13.2, β1::Real=4.7,
                          σ2_u::Real=0.175, σ2_A::Real=7.2)
    u = rand(Normal(0, sqrt(σ2_u)), N)
    A = rand(Normal(0, sqrt(σ2_A)), N)
    S = π .+ A
    Y = β1 .+ β2 .* S .+ β3 .* A .+ u
    return S, Y
end
```

Modify this function in the cell below. You need to add `β4` as a positional argument, draw `W`, include the `β4 .* W` term, and return `S, W, Y`.

In [ ]:
"""
    schooling_sample(β2, β3, β4, N; π=13.2, β1=4.7, σ2_u=0.175, σ2_A=7.2)

Generate a random sample from the extended schooling DGP with experience.

# Arguments (positional)
- `β2`: Return to schooling
- `β3`: Effect of ability on earnings (set to 0 for no endogeneity)
- `β4`: Return to work experience
- `N`: Sample size

# Keyword arguments (calibration parameters)
- `π`: Mean schooling (default 13.2)
- `β1`: Intercept (default 4.7)
- `σ2_u`: Variance of idiosyncratic error (default 0.175)
- `σ2_A`: Variance of ability (default 7.2)

# Returns
- `S`: Vector of schooling values
- `W`: Vector of experience values
- `Y`: Vector of log earnings
"""
function schooling_sample(β2::Real, β3::Real, β4::Real, N::Integer;
                          π::Real=13.2, β1::Real=4.7,
                          σ2_u::Real=0.175, σ2_A::Real=7.2)

    # YOUR CODE HERE
    # Hint: Start from the week 9 version above. You need to:
    #   1. Draw W from DiscreteUniform(0, 20) using rand(DiscreteUniform(0, 20), N)
    #   2. Add β4 .* W to the earnings equation
    #   3. Return S, W, Y

    error("Not yet implemented")
end

In [ ]:
# Generate one sample with calibrated values
Random.seed!(42)
S, W, Y = nothing

# Form the regressor matrix X = [1  S  W], hint: use hcat
X = nothing  # YOUR CODE HERE

## Exercise 2

The log-likelihood function for the normal regression model is

$$
L(\beta, \sigma_e^2) = -\frac{N}{2} \ln \left(2 \pi \sigma_e^2 \right) 
-\frac{1}{2 \sigma_e^2} \sum_{i=1}^N (y_i - x_i' \beta)^2
$$

where $x_i$ is the $i$-th row of the regressor matrix $X$.

1. Write a function `log_likelihood(y, X, β, σ2)` that computes this formula.
2. Write a closure `neg_log_likelihood_closure(y, X)` that returns the *negative* log-likelihood as a function of a single parameter vector $\theta = (\beta', \sigma_e^2)'$ only (as in week 10).
3. Use `Optim.optimize` to find the MLE.

Hint: The parameter vector $\theta$ has 4 entries: $\theta = (\beta_1, \beta_2, \beta_4, \sigma_e^2)'$. In the closure, unpack it as `θ[1:end-1]` for $\beta$ and `θ[end]` for $\sigma_e^2$.

In [ ]:
"""
    log_likelihood(y, X, β, σ2)

Compute the log-likelihood of the normal regression model.

Uses the formula L(β, σ²) = -N/2 * ln(2πσ²) - 1/(2σ²) Σ(yᵢ - xᵢ'β)².

# Arguments
- `y::Vector`: dependent variable (log earnings)
- `X::Matrix`: N×K regressor matrix (includes intercept column)
- `β::Vector`: K-vector of regression coefficients
- `σ2::Real`: error variance (must be positive)

# Returns
- `Float64`: the log-likelihood value
"""
function log_likelihood(y, X, β, σ2)

    # YOUR CODE HERE

    error("Not yet implemented")
end

In [ ]:
"""
    neg_log_likelihood_closure(y, X)

Return a closure that computes the negative log-likelihood as a function of θ only.

The parameter vector θ = [β; σ²] stacks the regression coefficients and the error
variance into a single vector suitable for numerical optimization.

# Arguments
- `y::Vector`: dependent variable (captured by the closure)
- `X::Matrix`: regressor matrix (captured by the closure)

# Returns
- `Function`: a function θ -> -log_likelihood(y, X, θ[1:end-1], θ[end])
"""
neg_log_likelihood_closure(y, X) = nothing  # YOUR CODE HERE: replace nothing with a closure

# Initialize the closure with our sample data
neg_ll = nothing  # YOUR CODE HERE

# Initial guess 
# note: could use OLS here but we know from lecture that MLE = OLS in this model
# so using OLS is a bit silly here, so just go with 0.5 for all 4 parameters
θ_ini = nothing  # YOUR CODE HERE

# Optimize
result = nothing  # YOUR CODE HERE: use optimize(neg_ll, θ0)

# Extract estimates
β_hat = nothing   # θ_hat[1:3]
σ2_hat = nothing  # θ_hat[4]

## Exercise 3

The asymptotic result for the MLE is

\begin{align*}
    \sqrt{N}
    \begin{pmatrix}
        \hat{\beta}^\text{ML} - \beta \\
        \hat{\sigma}_e^{2,\text{ML}} - \sigma_e^2
    \end{pmatrix}
    \overset{d}{\to}
    \mathcal{N}
    \begin{pmatrix}
        \begin{pmatrix}
            0\\0
        \end{pmatrix}
        ,
        \begin{pmatrix}
            \sigma_e^2 \, E(X_i X_i')^{-1} &
            0 \\
            0' &
            2\sigma_e^4
        \end{pmatrix}
\end{pmatrix}.
\end{align*}

This implies that the finite-sample variance of $\hat{\beta}^\text{ML}$ is approximately

$$\text{Var}(\hat{\beta}^\text{ML}) \approx \sigma_e^2 \, (X'X)^{-1}$$

Obtain estimates of the standard errors of $\hat{\beta}^\text{ML}$ using the analytical formula above.

Hint: The standard errors are the square roots of the diagonal entries of the variance-covariance matrix.

In [1]:
# Analytical standard errors
Avar_β = nothing  # variance-covariance matrix: σ2_hat * inv(X' * X)
se_β = nothing    # standard errors: sqrt.(diag(Avar_β))

# YOUR CODE HERE

## Exercise 4

Instead of using the **analytical** Hessian matrix to obtain the asymptotic variance, the `Optim` package together with the `NLSolversBase` package can **numerically** approximate the Hessian.

Using your closure `neg_ll`, the following code computes the numerical Hessian at the MLE and extracts standard errors:

In [ ]:
using NLSolversBase

td = TwiceDifferentiable(neg_ll, ones(4))
H = hessian!(td, θ_hat)

# Standard errors from numerical Hessian (all 4 parameters including σ²)
se_numerical = sqrt.(diag(inv(H)))

@show se_β
@show se_numerical

Are the standard errors from the numerical Hessian similar to the analytical standard errors from Exercise 3?

Note: The numerical Hessian gives standard errors for all 4 parameters $(\beta_1, \beta_2, \beta_4, \sigma_e^2)$, while the analytical formula in Exercise 3 only covers the 3 entries of $\beta$. The theoretical standard error of $\hat{\sigma}_e^2$ is $\sqrt{2\sigma_e^4 / N}$.